### 발표 평가 요소 구분 모듈 -- 자연어

#### 환경 설정

In [8]:
# library import
import os
from dotenv import load_dotenv
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 환경 변수
load_dotenv()
BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL")

#### Structured Output 정의

In [9]:
# 발표 평가 요소 모델 정의
# 주의: json_schema strict 모드는 dict[str, ...]처럼 키가 고정되지 않은 필드를 지원하지 않으므로
# 평가 요소를 고정 필드로 갖는 모델로 표현
class Criteria(BaseModel):
	speed: str | None = Field(description="발표 속도")
	pause: str | None = Field(description="지나치게 긴 침묵 횟수")
	volume: str | None = Field(description="발표 볼륨")
	filler: str | None = Field(description="발표 중 filler 단어 사용 횟수")
	gaze: str | None = Field(description="발표 중 시선 처리")
	script_used: bool | None = Field(description="스크립트 사용 여부")
	script_dependency: str | None = Field(description="스크립트 의존도")
	script_similarity: str | None = Field(description="스크립트와 발표 내용의 유사도")


class EvaluationCriteriaAnalysis(BaseModel):
	display_criteria: list[str] = Field(description="사용자 입력 표현을 살려 짧게 다듬은 평가 기준 목록 (예: \"추임새 5번 이하\")")
	values: Criteria = Field(description="평가 요소별 기준 값 (기준이 없으면 null)")
	excluded: list[str] = Field(description="평가 기준으로 사용하지 않은 평가 항목 이름(명사구) 목록 (예: \"발표 시간\")")

#### System Prompt 작성

In [10]:
SYSTEM_PROMPT = """
너는 발표 평가 기준을 분석하는 전문가야.

입력은 사용자가 자유로운 문장으로 작성한 발표 평가 기준이야.
입력에서 발표 분석 시스템이 측정할 수 있는 평가 요소만 골라 기준 값을 추출하고(values),
사용자에게 보여줄 짧은 평가 기준 문장 목록(display_criteria)을 만들어.
평가 기준으로 사용하지 않은 입력 부분은 excluded에 모아.

## 평가 요소 (values 필드)

시스템이 측정할 수 있는 평가 요소는 아래 8개뿐이야.
입력 표현이 달라도 의미가 같으면 해당 필드로 매핑해.

| 필드 | 의미 | 입력에서 나올 수 있는 표현 |
|---|---|---|
| speed | 말하기 속도 | 말 속도, 빠르기, 템포, 천천히/빠르게 |
| pause | 지나치게 긴 침묵 | 침묵, 공백, 말 끊김, 머뭇거림, 정적 |
| volume | 목소리 크기 | 음량, 성량, 목소리 크기, 잘 들리게 |
| filler | 채움말 사용 | 추임새, 채움말, 군말, 필러, "음", "어", "그" |
| gaze | 시선 처리 | 시선, 아이컨택, 청중 응시, 화면만 보지 않기 |
| script_used | 대본 사용 허용 여부 | 대본/스크립트/원고를 봐도 되는지 |
| script_dependency | 발표 중 대본을 보는 정도 | 대본 의존, 대본을 그대로 읽기, 참고만 하기 |
| script_similarity | 실제 발화 내용과 대본 내용의 일치도 | 대본 내용대로 말하기, 준비한 내용 빠짐없이 전달 |

## values 작성 규칙

* 입력에 해당 요소에 대한 언급이 없으면 반드시 null로 둬. 추측해서 채우지 마.
* script_used는 boolean이야.
  * "대본 없이", "대본을 보면 안 된다" → false
  * "대본을 봐도 된다", "대본은 참고만" → true
  * 대본 사용 여부를 판단할 단서가 없으면 null로 둬.
* 나머지 필드는 기준을 짧은 명사구로 적어. 문장 전체를 복사하지 마.
  * 숫자와 단위는 입력 그대로 유지해. (예: "5회 이하", "3초 이상 침묵 2회 이하")
  * 입력에 숫자가 없는 모호한 기준은 숫자를 지어내지 말고 입력의 표현을 간결하게 남겨. (예: "너무 빠르지 않게", "줄이기")
* 같은 요소가 여러 번 언급되면 하나로 합치고, 더 구체적인 기준을 우선해.

## 평가 대상이 아닌 기준

아래처럼 8개 필드에 해당하지 않는 기준은 values와 display_criteria 어디에도 넣지 말고 excluded에 넣어.

* 발표 시간, 억양, 발음, 자세, 제스처, 표정
* 발표 내용, 논리 구성, 주제의 창의성, 독창성, 다른 팀과의 차별성
* 발표 자료의 디자인이나 신뢰도
* 질의응답, 태도, 팀워크

억양처럼 목소리와 관련 있어 보여도 위 8개 필드의 의미와 다르면 매핑하지 마.

## display_criteria 작성 규칙

* values에서 null이 아닌 필드마다 문장을 정확히 하나씩 만들어. null인 필드는 넣지 마.
* 순서는 사용자가 입력한 순서를 따라.
* 사용자가 입력한 표현을 최대한 그대로 살려. 입력에 쓰인 단어(예: "추임새", "목소리", "말이 빠르지")를 다른 용어(채움말, 음량, 말하기 속도 등)로 바꾸지 마.
* 입력 문장에서 해당 기준 부분만 떼어 20자 이내로 다듬어. "~해야 합니다", "~해주세요" 같은 어미와 마침표는 빼고 "~하기", "~않게", "~이하"처럼 짧게 끝내.
* 영어 필드명은 쓰지 마.
* 값의 의미를 바꾸거나 기준을 더하지 마.

## excluded 작성 규칙

* 입력 중 values에 반영하지 않은 부분을 빠짐없이 넣어. 입력의 모든 기준은 values 또는 excluded 중 한 곳에 반드시 들어가야 해.
* 기준의 내용(숫자, 정도, 조건, "~해야 한다" 같은 요구)은 빼고, 무엇을 평가하는지 나타내는 항목 이름만 필드명처럼 짧은 명사구로 넣어.
  * 예: "발표 시간은 5분이며 ±30초를 허용합니다" → "발표 시간"
  * 예: "발표 자료의 내용은 신뢰할 수 있어야 합니다" → "발표 자료 신뢰도"
  * 예: "다른 팀과 비슷한 주제는 피해야 합니다" → "주제 차별성"
  * 예: "발표 내용은 이해하기 쉽고 논리적으로 구성되어야 합니다" → "발표 내용 이해도", "논리 구성"
* 같은 항목을 가리키는 기준이 여러 번 나오면 하나로 합쳐. (예: "발표 시간은 10분이며 시간을 지켜야 합니다" → "발표 시간")
* 한 문장에 반영한 기준과 반영하지 않은 기준이 섞여 있으면, 반영하지 않은 기준 부분만 떼어서 넣어.
  * 예: "적절한 음량과 억양을 사용해야 합니다." → 음량은 volume에 반영하고, excluded에는 "억양"만 넣어.
* 한 문장에 반영하지 않은 기준이 여러 개 있으면 기준마다 따로 나눠 넣어.
* 입력 순서를 유지해.
* 모든 기준이 values에 반영되었으면 빈 리스트([])로 둬.

## 예시

입력: "추임새는 5번 이하로 해주시고, 대본 없이 발표해야 합니다. 발표 자료 디자인도 깔끔해야 합니다."
출력:
{
  "display_criteria": ["추임새 5번 이하", "대본 없이 발표하기"],
  "values": {
    "speed": null, "pause": null, "volume": null, "filler": "5회 이하", "gaze": null,
    "script_used": false, "script_dependency": null, "script_similarity": null
  },
  "excluded": ["발표 자료 디자인"]
}

입력: "말이 너무 빠르지 않게 하고 목소리는 뒷자리까지 들리게 해주세요. 대본은 참고만 하되 준비한 내용은 빠짐없이 전달해야 합니다. 발표 시간은 7분입니다."
출력:
{
  "display_criteria": ["말이 너무 빠르지 않게", "목소리 뒷자리까지 들리게", "대본 참고 허용", "대본은 참고만 하기", "준비한 내용 빠짐없이 전달하기"],
  "values": {
    "speed": "너무 빠르지 않게", "pause": null, "volume": "뒷자리까지 들리게", "filler": null, "gaze": null,
    "script_used": true, "script_dependency": "참고 수준", "script_similarity": "준비한 내용 빠짐없이 전달"
  },
  "excluded": ["발표 시간"]
}

입력: "발표 주제는 창의적이어야 하고 질의응답에 성실히 답해야 합니다."
출력:
{
  "display_criteria": [],
  "values": {
    "speed": null, "pause": null, "volume": null, "filler": null, "gaze": null,
    "script_used": null, "script_dependency": null, "script_similarity": null
  },
  "excluded": ["주제 창의성", "질의응답"]
}
"""

#### LLM 모델 불러오기

In [11]:
os.getenv('OPENAI_MODEL'), os.getenv('OPENAI_BASE_URL')

('openai/gpt-5.6-luna',
 'https://mlapi.run/286e9158-d32e-436d-a23d-36b43fc8e68a/v1')

In [12]:
# LLM 객체 생성
model = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# Structured Output 연결
# 자유 키 dict 없이 고정 필드(display_criteria/values)만 사용하므로
# json_schema strict 모드(기본값)를 그대로 사용 가능 (function tools가 아니라 response_format 기반이라
# reasoning_effort와 충돌하지 않음)
structured_output = model.with_structured_output(EvaluationCriteriaAnalysis)

# 실제 모델 호출이 되는지 간단한 테스트
test_input = "발표는 10분 정도 진행하며 시간을 지켜야 합니다. 발표 중 불필요한 추임새를 줄이고 적절한 음량과 억양을 사용해야 합니다. 발표 자료의 내용은 신뢰할 수 있어야 하고 질의응답에도 적극적으로 참여해야 합니다."
response = structured_output.invoke(test_input)
response

EvaluationCriteriaAnalysis(display_criteria=['발표 시간 10분 준수', '불필요한 추임새 줄이기', '적절한 음량과 억양 사용'], values=Criteria(speed='발표 시간 약 10분을 준수함', pause=None, volume='청중이 잘 들을 수 있는 적절한 음량과 자연스러운 억양을 사용함', filler='불필요한 추임새를 최소화함', gaze=None, script_used=None, script_dependency=None, script_similarity=None), excluded=['발표 자료의 신뢰성', '질의응답 참여도'])

#### 단일 평가 기준 test

In [13]:
criteria_input = "발표 시간은 10분이며 시간을 지켜야 합니다. 발표 중 불필요한 추임새를 줄이고 적절한 음량과 억양을 사용해야 합니다. 발표 자료의 내용은 신뢰할 수 있어야 하고 질의응답에도 적극적으로 참여해야 합니다."

result = structured_output.invoke(
    [
        ("system", SYSTEM_PROMPT),
        ("user", criteria_input)
    ]
)
data = result.model_dump()
data

{'display_criteria': ['불필요한 추임새 줄이기', '적절한 음량'],
 'values': {'speed': None,
  'pause': None,
  'volume': '적절한 음량',
  'filler': '줄이기',
  'gaze': None,
  'script_used': None,
  'script_dependency': None,
  'script_similarity': None},
 'excluded': ['발표 시간', '억양', '발표 자료 신뢰도', '질의응답']}

In [14]:
criteria_inputs = ["시간은 5분이며 ±30초를 허용합니다. 대본은 사용하지 않아야 하고, 발표자의 시선은 청중을 향해야 합니다.",
"발표는 시간을 엄수해야 하며, 말하는 속도도 너무 빠르지 않도록 주의해야 합니다. 발표 내용은 이해하기 쉽고 논리적으로 구성되어야 합니다.",
"발표 시간은 10분 이내로 해야 합니다. 대본을 보면서 발표해도 되지만, 지나치게 의존해서는 안 됩니다. 발표자의 자세도 단정해야 합니다.",
"발표 주제는 창의적이어야 하며 우리 팀의 Agent 활용이 잘 드러나야 합니다. 다른 팀과 비슷한 주제는 피해야 합니다. 발표 시간은 7분입니다.",
"발표 시간은 10분이며 시간을 지켜야 합니다. 발표 중 불필요한 추임새를 줄이고 적절한 음량과 억양을 사용해야 합니다. 발표 자료의 내용은 신뢰할 수 있어야 하고 질의응답에도 적극적으로 참여해야 합니다."
]

for i in range(len(criteria_inputs)):
    result = structured_output.invoke(
        [
            ("system", SYSTEM_PROMPT),
            ("user", criteria_inputs[i])
        ]
    )
    data = result.model_dump()
    print(f"Input {i+1}: {criteria_inputs[i]}")
    print(f"Output {i+1}: {data}\n")

Input 1: 시간은 5분이며 ±30초를 허용합니다. 대본은 사용하지 않아야 하고, 발표자의 시선은 청중을 향해야 합니다.
Output 1: {'display_criteria': ['대본은 사용하지 않기', '시선은 청중을 향하기'], 'values': {'speed': None, 'pause': None, 'volume': None, 'filler': None, 'gaze': '청중을 향하기', 'script_used': False, 'script_dependency': None, 'script_similarity': None}, 'excluded': ['발표 시간']}

Input 2: 발표는 시간을 엄수해야 하며, 말하는 속도도 너무 빠르지 않도록 주의해야 합니다. 발표 내용은 이해하기 쉽고 논리적으로 구성되어야 합니다.
Output 2: {'display_criteria': ['말하는 속도 너무 빠르지 않게'], 'values': {'speed': '너무 빠르지 않게', 'pause': None, 'volume': None, 'filler': None, 'gaze': None, 'script_used': None, 'script_dependency': None, 'script_similarity': None}, 'excluded': ['발표 시간', '발표 내용 이해도', '논리 구성']}

Input 3: 발표 시간은 10분 이내로 해야 합니다. 대본을 보면서 발표해도 되지만, 지나치게 의존해서는 안 됩니다. 발표자의 자세도 단정해야 합니다.
Output 3: {'display_criteria': ['대본 보면서 발표 허용', '대본에 지나치게 의존하지 않기'], 'values': {'speed': None, 'pause': None, 'volume': None, 'filler': None, 'gaze': None, 'script_used': True, 'script_dependency': '지나치게 의존하지 않기', 'script_similarit